In [2]:
# 4_feature_eng_ukhls.ipynb


# Applies feature engineering to the backfilled UKHLS Wave O data.
# All operations are driven by config_variables.py — no hardcoded lists.
# Steps:
#   1. Load o_indresp_backfilled.pkl
#   2. Apply value recodes (RECODE_MAPS)     e.g. hiqual_dv 9 → 5
#   3. Apply floor clipping  (FLOOR_VALUES)  e.g. payn_dv < 0 → 0
#   4. Apply upper clipping  (CLIP_VALUES)   e.g. carmiles > 50,000 → 50,000
#   5. One-hot encode categorical variables  (ONE_HOT_VARS) — originals kept
#   6. Force ALL columns to float32 (includes OHE columns added in step 5)
#   7. Save as o_indresp_feature_eng.pkl


import sys, os
sys.path.insert(0, os.path.abspath('..'))


import importlib
from data_pipeline.config_paths import USE_TEST_DATA, DATA_FOLDER
import data_pipeline.config_variables as _ukhls_vars
importlib.reload(_ukhls_vars)


import pandas as pd
import numpy as np


from data_pipeline.config_variables import (
    RECODE_MAPS, FLOOR_VALUES, CLIP_VALUES, ONE_HOT_VARS, VARIABLES, TRANSFORMS,
 )


# ── Config ────────────────────────────────────────────────────────────────────
WAVE       = "o"
INPUT_PKL  = f"../{DATA_FOLDER}/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"
OUTPUT_PKL = f"../{DATA_FOLDER}/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"


def get_base_code(col_name, wave_prefix):
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name


# ── 1. Load ───────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_PKL} ...")
df = pd.read_pickle(INPUT_PKL)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")


# ── 1b. Apply transforms (e.g. birth year → age) ──────────────────────────
if TRANSFORMS:
    print("\nStep 1b: Applying transforms ...")
    for base, tfm in TRANSFORMS.items():
        col = f"{WAVE}_{base}"
        if col not in df.columns:
            print(f"  SKIP {col} — not in dataframe")
            continue
        if tfm == "birth_year_to_age":
            current_year = pd.Timestamp.now().year
            raw = pd.to_numeric(df[col], errors='coerce')
            df[col] = current_year - raw
            df.loc[df[col] < 0, col] = np.nan
            print(f"  {col}: birth_year_to_age (current_year={current_year})")
        else:
            print(f"  WARNING: unknown transform '{tfm}' for {col} — skipping")


# ── 2. Value Recodes ──────────────────────────────────────────────────────────
print("\nStep 2: Applying value recodes ...")
recoded = []
for col in df.columns:
    base = get_base_code(col, WAVE)
    recode = RECODE_MAPS.get(base)
    var_def = VARIABLES.get(base, {})
    categories = set((var_def.get('categories') or {}).keys())
    if recode:
        # Check for values not in categories and not remapped
        col_numeric = pd.to_numeric(df[col], errors='coerce')
        unique_vals = set(col_numeric.dropna().unique())
        mapped_vals = set(recode.keys())
        allowed_vals = categories | mapped_vals
        not_listed = unique_vals - allowed_vals
        if not_listed:
            not_listed_sorted = sorted(not_listed)
            not_listed_str = ', '.join(str(x) for x in not_listed_sorted)
            print(f"WARNING: {col} has values not in categories or recode map: {not_listed_str}")
        df[col] = col_numeric.replace(recode)
        recoded.append(f"  {col}: {recode}")
if recoded:
    print("\n".join(recoded))
else:
    print("  (none)")


# ── 3. Floor Clipping ─────────────────────────────────────────────────────────
print("\nStep 3: Applying floor values ...")
for col in df.columns:
    base = get_base_code(col, WAVE)
    floor_val = FLOOR_VALUES.get(base)
    if floor_val is not None:
        numeric = pd.to_numeric(df[col], errors='coerce')
        n_floored = int((numeric < floor_val).sum())
        df[col] = numeric.clip(lower=floor_val)
        print(f"  {col}: floored at {floor_val}  ({n_floored:,} rows affected)")


# ── 4. Upper Clipping ─────────────────────────────────────────────────────────
print("\nStep 4: Applying clip values ...")
for col in df.columns:
    base = get_base_code(col, WAVE)
    clip_val = CLIP_VALUES.get(base)
    if clip_val is not None:
        numeric = pd.to_numeric(df[col], errors='coerce')
        n_clipped = int((numeric > clip_val).sum())
        df[col] = numeric.clip(upper=clip_val)
        print(f"  {col}: clipped at {clip_val:,}  ({n_clipped:,} rows affected)")


# ── 4b. Derive binary features from config ─────────────────────────────────────
# Dynamically derive binary features as specified in config_variables.py
print("\nStep 4b: Deriving binary features from config ...")
for base, var_def in VARIABLES.items():
    derrived_cfg = var_def.get('create_binary_derrived_feature')
    if derrived_cfg:
        src_col = f"{WAVE}_{base}"
        derrived_col = f"{WAVE}_{derrived_cfg['feature_name']}"
        threshold = derrived_cfg.get('threshold', 0)
        if src_col in df.columns:
            # Binary: 1 if value > threshold, else 0
            df[derrived_col] = (pd.to_numeric(df[src_col], errors='coerce') > threshold).astype(np.float32)
            n_with = int((df[derrived_col] == 1.0).sum())
            print(f"  {derrived_col}: {n_with:,} rows > {threshold} ({100*n_with/len(df):.1f}%)")
        else:
            print(f"  SKIP — {src_col} not in dataframe")


# ── 4c. Immigrant generation (derived from ukborn, pacob, macob) ───────────────
# 1 = born outside UK; 2 = UK-born with ≥1 parent born abroad (codes ≥5); 3 = both parents UK (1–4);
# 0 = missing ukborn or UK-born but parent info insufficient to classify.
print("\nStep 4c: Deriving immigrant_gen ...")
_uk_col = f"{WAVE}_ukborn"
_pf_col = f"{WAVE}_pacob"
_pm_col = f"{WAVE}_macob"
_out_col = f"{WAVE}_immigrant_gen"
if _uk_col in df.columns and _pf_col in df.columns and _pm_col in df.columns:
    u = pd.to_numeric(df[_uk_col], errors="coerce")
    pf = pd.to_numeric(df[_pf_col], errors="coerce")
    pm = pd.to_numeric(df[_pm_col], errors="coerce")
    inv = {-9.0, -8.0, -7.0, -2.0, -1.0}
    inv_pf = pf.isna() | pf.isin(list(inv))
    inv_pm = pm.isna() | pm.isin(list(inv))
    first_gen = u == 5.0
    uk_born = (u >= 1.0) & (u <= 4.0)
    pf_uk = (pf >= 1.0) & (pf <= 4.0) & ~inv_pf
    pm_uk = (pm >= 1.0) & (pm <= 4.0) & ~inv_pm
    pf_abroad = (pf >= 5.0) & ~inv_pf
    pm_abroad = (pm >= 5.0) & ~inv_pm
    gen3 = uk_born & pf_uk & pm_uk
    gen2 = uk_born & (pf_abroad | pm_abroad)
    gen = pd.Series(0.0, index=df.index, dtype=np.float64)
    gen = gen.mask(first_gen, 1.0)
    gen = gen.mask(gen3, 3.0)
    gen = gen.mask(gen2, 2.0)
    df[_out_col] = gen.astype(np.float32)
    vc = df[_out_col].value_counts().sort_index()
    print(f"  {_out_col}: " + ", ".join(f"{int(k)}→{int(v):,}" for k, v in vc.items()))
else:
    print(f"  SKIP {_out_col} — need {_uk_col}, {_pf_col}, {_pm_col}")


# ── 5. One-hot encoding ───────────────────────────────────────────────────────
# For each variable with one_hot defined, create binary indicator columns (bool).
# Original column is kept unchanged.
# Naming: {wave}_{base}_{int(code)}  e.g. o_jbstat_2 = 1 if employed
# Type casting to float32 is handled centrally in step 6.
print("\nStep 5: One-hot encoding ...")
oh_new_cols = []
for base, one_hot_spec in ONE_HOT_VARS.items():
    col = f"{WAVE}_{base}"
    if col not in df.columns:
        print(f"  SKIP {col} — not in dataframe")
        continue


    src = pd.to_numeric(df[col], errors='coerce')


    # Determine which codes to encode
    if one_hot_spec is True:
        # Prefer group_labels keys (post-recode canonical codes) when present;
        # fall back to categories keys so we don't create zero-filled columns
        # for raw codes that have been recoded away.
        var_def = VARIABLES[base]
        codes = list((var_def.get("group_labels") or var_def["categories"]).keys())
    else:
        # one_hot_spec is a list of specific codes
        codes = list(one_hot_spec)


    for code in codes:
        new_col = f"{WAVE}_{base}_{int(code)}"
        df[new_col] = (src == code)   # bool for now; step 6 casts to float32
        oh_new_cols.append(new_col)
        print(f"  {new_col}  ({int((df[new_col]).sum()):,} = 1)")


print(f"\n  {len(oh_new_cols)} binary indicator columns added; originals kept.")


# ── 6. Force ALL columns to float32 ──────────────────────────────────────────
# Runs after step 5, so OHE columns added above are included.
print("\nStep 6: Converting all features to float32 ...")
for col in df.columns:
    if col == 'pidp':
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.int64)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)


n_nulls = df.drop(columns=['pidp']).isna().sum().sum()
if n_nulls > 0:
    print(f"  WARNING — {n_nulls:,} NaN values remain after feature engineering")
    print(df.drop(columns=['pidp']).isna().sum()[lambda s: s > 0])
else:
    print("  No NaN values — clean feature matrix")
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)
# ── 7. Save ───────────────────────────────────────────────────────────────────
df.to_pickle(OUTPUT_PKL, protocol=5)
print(f"\nDone. Feature matrix saved to {OUTPUT_PKL}")
print(f"Shape: {df.shape}")
print(df.dtypes)
print(df.dtypes)



Loading ../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl ...
Loaded 32,849 rows × 29 columns

Step 1b: Applying transforms ...
  o_doby_dv: birth_year_to_age (current_year=2026)

Step 2: Applying value recodes ...
Filling 26875 missing values in o_englang with 1.0 (config fill)
  o_jbstat: {1.0: 1.0, 2.0: 1.0, 12.0: 1.0, 13.0: 1.0, 3.0: 3.0, 4.0: 4.0, 5.0: 5.0, 6.0: 5.0, 14.0: 5.0, 15.0: 5.0, 7.0: 7.0, 9.0: 7.0, 11.0: 7.0, 8.0: 8.0, 10.0: 8.0, 97.0: 8.0, -1.0: 8.0}
  o_netpusenew: {-8.0: 7.0, -1.0: 7.0}
  o_racel_dv: {-9.0: 0.0, 1.0: 1.0, 2.0: 1.0, 4.0: 1.0, 5.0: 1.0, 6.0: 1.0, 7.0: 1.0, 8.0: 1.0, 9.0: 2.0, 10.0: 3.0, 11.0: 3.0, 12.0: 4.0, 13.0: 4.0, 17.0: 5.0, 14.0: 6.0, 15.0: 7.0, 16.0: 7.0, 97.0: 8.0}
  o_hiqual_dv: {9.0: 6.0}
  o_jbnssec8_dv: {-9.0: 0.0, -8.0: 0.0, -7.0: 0.0, -2.0: 0.0, -1.0: 0.0}
  o_englang: {-9.0: 1.0, -8.0: 1.0, -7.0: 1.0, -2.0: 1.0, -1.0: 1.0}

Step 3: Applying floor values ...
  o_carmiles: floored at 0  (8,655 rows affected)
  o_fimngrs_dv: floored at